# APM_50441_EP - Machine Learning 2 - Lab 7: Machine learning at scale

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/APM-50441-EP/APM_50441_EP-2026-2027/blob/main/PC7/APM_50441_EP_Lab_7_simplified_with_solution.ipynb) or run it on your own computer (see *Working environment* below).

In [ ]:
# On Colab (R runtime) only: installs the packages of this lab (once per session, a few minutes).
# On your own computer, use PC/env/install_packages.R instead (see Working environment).
if (nzchar(Sys.getenv("COLAB_RELEASE_TAG"))) {
  ubuntu <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", readLines("/etc/os-release"), value = TRUE))
  options(repos = c(CRAN = paste0("https://packagemanager.posit.co/cran/__linux__/", ubuntu, "/latest")))  # binaries for this Ubuntu
  source("https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/env/install_packages.R")
  install_lab("lab7")
}

This lab illustrates the seventh lecture (*At Scale*) on the flights of the United States in 2015. Its goals are:

- to read and store a large dataset efficiently (binary columnar formats, Parquet, partitions), and to use a sample when it is enough,
- to compare engines for the split/apply/combine computations on the full data, in memory (`dplyr`, `data.table`, `duckdb`) and directly on files (`arrow`, `duckdb`),
- to profile and optimize code, from the algorithm to the compiled language (`profvis`, `bench`, `Rcpp`),
- to parallelize a cross validation, and to measure when it pays off (`mirai`, `purrr::in_parallel()`),
- to serve a model with a REST API (`plumber2`) and a web application (`shiny` and `bslib`).

The optional *To go further* part, after the core, can be skipped at first: it covers Map/Reduce by hand, Spark, larger-than-memory computations, the memory hierarchy, pipelines, in-database scoring and the deployment of models.

::: {.no-handout}
We use `R` with the `tidyverse`. Most of the tools of this lab are not in the slides, which focus on the concepts: they are the current (2026) R implementations of these concepts.

**Working environment.** Colab is the simplest option: open the notebook version of this lab (`APM_50441_EP_Lab_7.ipynb`, *Open in Colab* link) with the R runtime, and run its first cell, which installs the packages. To work on your own computer, download the lab folder (the `.qmd` file and its files) or clone the course repository, open the `.qmd` file in RStudio or Positron (or VS Code with the R extension), and run it chunk by chunk or render it with Quarto. The packages are installed once with the script `PC/env/install_packages.R` (`source("install_packages.R")`, then `install_lab("lab7")`); Spark, only used in a *To go further* block, is installed separately with `install_spark()`. The data are read from the local files when they are present (otherwise they are downloaded).

**Use of AI.** AI assistants are allowed, but you must be able to explain every line you use, and you should answer the *predict before you run* questions before running the code: the suggested code may not be correct!
:::

In [ ]:
library(tidyverse)
library(data.table)
library(arrow)
library(duckdb)
theme_set(theme_light())
if (!file.exists("data/flights_sample.csv.gz")) {
  dir.create("data", showWarnings = FALSE)
  for (f in c("flights_sample.csv.gz", "airlines.csv")) {
    download.file(paste0("https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/PC7/data/", f), file.path("data", f), mode = "wb")
  }
}
for (f in c("cumsum.cc", "flights_api.R", "flights_app.R")) {  # the code files of the lab (Parts 3 and 5)
  if (!file.exists(f)) download.file(paste0("https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/PC7/", f), f, mode = "wb")
}
airlines <- read_csv("data/airlines.csv", show_col_types = FALSE)

The full dataset ($5.8$ million flights, $1.4$ GB in memory) is too large for the computations of Parts 1, 3, 4 and 5 on some laptops: they use a random sample of $10\%$ of the flights ($582\,000$ flights). Part 2 uses the full data, stored as Parquet files ($138$ MB, downloaded once): it needs about $4$ GB of memory, which Colab and most laptops have.

## 1. Reading, storing and sampling

**Question** *Predict before you run:* which reader is the fastest? Read `data/flights_sample.csv.gz` with `readr::read_csv`, `data.table::fread` and `arrow::read_csv_arrow`, and compare their times with `bench::mark` ($3$ iterations, `check = FALSE`). What is the size of the data in memory (`lobstr::obj_size`)?

**Answer**

In [ ]:
col_types <- cols(ORIGIN_AIRPORT = "c", DESTINATION_AIRPORT = "c", SCHEDULED_DEPARTURE = "c", DEPARTURE_TIME = "c",
                  WHEELS_OFF = "c", WHEELS_ON = "c", SCHEDULED_ARRIVAL = "c", ARRIVAL_TIME = "c", CANCELLATION_REASON = "c")
reading <- bench::mark(
  readr = read_csv("data/flights_sample.csv.gz", col_types = col_types),
  data.table = fread("data/flights_sample.csv.gz"),
  arrow = read_csv_arrow("data/flights_sample.csv.gz"),
  iterations = 3, check = FALSE, filter_gc = FALSE)
reading |> select(expression, min, median, mem_alloc)
flights <- read_csv("data/flights_sample.csv.gz", col_types = col_types)
lobstr::obj_size(flights)

**Conclusion** `data.table::fread` and `arrow::read_csv_arrow` read the compressed CSV file in about $1$ s, $2$ to $2.5$ times faster than `readr` (the timings vary a little from one run to another): they parse the file with several threads. Arrow also allocates much less memory during the reading ($65$ MB against $170$ to $200$ MB). In memory, the $582\,000$ flights take $145$ MB: the full dataset (ten times more rows) takes about $1.4$ GB (PC7, *Back-of-the-envelope estimates*). Decompression, parsing and disk access all contribute. These end-to-end timings do not measure their individual costs.

---

**Question** Write the flights as a Parquet dataset partitioned by `MONTH` (`arrow::write_dataset` in `data/flights_parquet`), and compare the size on disk with the CSV file. Then open it with `open_dataset`, and compute the mean arrival delay of the flights of December for each airline *without* loading the data in memory (`filter`, `group_by`, `summarise`, then `collect`). Compare the time with the same computation from the CSV file.

**Answer**

In [ ]:
write_dataset(flights, "data/flights_parquet", partitioning = "MONTH")
parquet_files <- list.files("data/flights_parquet", recursive = TRUE, full.names = TRUE)
c(`CSV (MB)` = file.size("data/flights_sample.csv.gz") / 1e6, `Parquet (MB)` = sum(file.size(parquet_files)) / 1e6,
  `Parquet files` = length(parquet_files))
flights_ds <- open_dataset("data/flights_parquet")
december <- function(data) data |> filter(MONTH == 12) |> group_by(AIRLINE) |>
  summarise(arrival_delay = mean(ARRIVAL_DELAY, na.rm = TRUE)) |> collect()
bench::mark(parquet = december(flights_ds),
            csv = december(read_csv("data/flights_sample.csv.gz", col_types = col_types)),
            iterations = 3, check = FALSE, filter_gc = FALSE) |> select(expression, min, median, mem_alloc)

**Conclusion** The Parquet dataset ($12$ files, one per month) is about as large on disk as the gzipped CSV ($28$ MB against $23$ MB, with the default *snappy* compression, which favors speed over size). But the query on December is about $20$ times faster ($0.1$ s against $2$ s) and allocates $35$ times less memory: Arrow only opens the partition `MONTH=12` (partition pruning), only reads the $3$ needed columns (columnar format), and the data are binary, with no parsing. The data are never fully loaded in R.

---

The *sampling trick* (lecture): for many questions, a random sample is enough, as the precision of an estimate depends on the size of the sample. Often, we do not need the full population to obtain a good enough estimate (PC7, *Reservoir sampling*).

**Question** Estimate the mean arrival delay of all the flights of the sample with random subsamples of $n = 100$, $1000$, $10\,000$ and $100\,000$ flights ($200$ subsamples each): compare the standard deviation of the estimates with $\sigma/\sqrt{n}$. Then compare a simple random subsample of $2004$ flights with a subsample *stratified* by month ($167$ flights each month, the monthly means being weighted by the proportions of the months).

**Answer**

In [ ]:
delays <- flights$ARRIVAL_DELAY[!is.na(flights$ARRIVAL_DELAY)]
set.seed(1)
precision <- map_dfr(c(100, 1000, 10000, 100000), function(n) {
  estimates <- replicate(200, mean(sample(delays, n)))
  tibble(n = n, `sd of the estimates` = sd(estimates), `sigma / sqrt(n)` = sd(delays) / sqrt(n),
         `finite population SE` = sd(delays) * sqrt((1 - n / length(delays)) / n))
})
precision
with_month <- flights |> filter(!is.na(ARRIVAL_DELAY)) |> select(MONTH, ARRIVAL_DELAY)
monthly_weights <- with_month |> count(MONTH) |> mutate(weight = n / sum(n))
stratified <- replicate(200, with_month |> group_by(MONTH) |> slice_sample(n = 167) |>
                          summarise(m = mean(ARRIVAL_DELAY)) |> left_join(monthly_weights, by = "MONTH") |>
                          summarise(sum(m * weight)) |> pull())
c(`simple random sampling` = sd(replicate(200, mean(sample(with_month$ARRIVAL_DELAY, 2004)))), stratified = sd(stratified))

**Conclusion** At $n=100{,}000$ the standard error for the fixed file mean is $0.1125$ minutes after correcting for sampling without replacement from $571{,}396$ observed delays. Inference about future flights requires further sampling assumptions. The independent-sampling benchmark is $\sigma/\sqrt{n}$ ($3.9$ for $n = 100$, $0.12$ for $n = 100\,000$): it depends on the size $n$ of the sample: we do not need the $5.8$ million flights to estimate the mean delay well. A sample of a few thousand flights gives the mean delay within a minute, enough to explore the data. Here, the stratification by month does not help ($0.85$ against $0.84$): the monthly means differ little compared with the huge variability of the delays within a month. Stratification pays off when the strata explain a large part of the variance.

---

## 2. Split, apply, combine

On the $10\%$ sample, every engine computes such aggregates in a few tens of milliseconds: the fixed costs (parsing the query, starting the threads, converting the results) dominate and hide the differences between the engines. This part uses the full data: $5.8$ million flights, stored as $12$ Parquet files (one per month, $138$ MB in all) downloaded once into `data/flights_full_parquet`.

**Question** Load the full data in memory (`open_dataset(...) |> collect()`), and measure the time and the size (`lobstr::obj_size`). Then compute the mean arrival delay and the proportion of flights arriving more than $15$ minutes late for each airline, with `dplyr`, with `data.table` and with `duckdb` (through `dbplyr`: register the table in a DuckDB database with `duckdb_register`, then use the same `dplyr` verbs and `collect()`). Check that the results agree, and compare the times and the memory with `bench::mark`. Run the same comparison on the $10\%$ sample: how does each engine scale? Compare also the mean delays estimated on the sample with the exact ones.

**Answer**

In [ ]:
if (!dir.exists("data/flights_full_parquet")) {  # the full data, downloaded once (12 files of about 12 MB)
  for (m in 1:12) {
    file <- sprintf("data/flights_full_parquet/MONTH=%d/part-0.parquet", m)
    dir.create(dirname(file), recursive = TRUE, showWarnings = FALSE)
    download.file(paste0("https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/PC7/", file), file, mode = "wb")
  }
}
options(arrow.use_altrep = FALSE)  # real R vectors (otherwise Arrow converts the columns lazily, when they are used)
start <- Sys.time()
flights_full <- open_dataset("data/flights_full_parquet") |> collect()
c(`loading time (s)` = as.numeric(difftime(Sys.time(), start, units = "secs")), `size (MB)` = as.numeric(lobstr::obj_size(flights_full)) / 1e6)

con <- dbConnect(duckdb())
by_airline_dplyr <- function(data) data |> group_by(AIRLINE) |>
  summarise(arrival_delay = mean(ARRIVAL_DELAY, na.rm = TRUE), late = mean(ARRIVAL_DELAY > 15, na.rm = TRUE))
by_airline_dt <- function(data_dt) data_dt[, .(arrival_delay = mean(ARRIVAL_DELAY, na.rm = TRUE),
                                               late = mean(ARRIVAL_DELAY > 15, na.rm = TRUE)), by = AIRLINE]
by_airline_duckdb <- function(data_db) data_db |> group_by(AIRLINE) |>
  summarise(arrival_delay = mean(ARRIVAL_DELAY, na.rm = TRUE), late = mean(as.numeric(ARRIVAL_DELAY > 15), na.rm = TRUE)) |>
  collect()
compare_engines <- function(data, name, iterations) {
  data_dt <- as.data.table(data)    # a copy
  duckdb_register(con, name, data)  # no copy: DuckDB reads the columns of the R data frame
  data_db <- tbl(con, name)
  results <- list(dplyr = by_airline_dplyr(data), data.table = by_airline_dt(data_dt), duckdb = by_airline_duckdb(data_db)) |>
    map(\(r) arrange(as_tibble(r), AIRLINE))
  agree <- c(`data.table agrees` = isTRUE(all.equal(results$dplyr, results$data.table, check.attributes = FALSE)),
             `duckdb agrees` = isTRUE(all.equal(results$dplyr, results$duckdb, check.attributes = FALSE)))
  timing <- bench::mark(dplyr = by_airline_dplyr(data), data.table = by_airline_dt(data_dt), duckdb = by_airline_duckdb(data_db),
                        iterations = iterations, check = FALSE, filter_gc = FALSE)
  list(results = results, agree = agree, timing = timing |> transmute(engine = as.character(expression), median, mem_alloc))
}
on_full <- compare_engines(flights_full, "flights_full", iterations = 5)
on_sample <- compare_engines(flights, "flights", iterations = 10)
rbind(`full data` = on_full$agree, `10% sample` = on_sample$agree)
bind_rows(`full data` = on_full$timing, `10% sample` = on_sample$timing, .id = "data") |>
  pivot_wider(names_from = data, values_from = c(median, mem_alloc)) |> print(width = Inf)
results <- on_full$results       # the exact statistics
flights_db <- tbl(con, "flights")  # the sample in DuckDB, used in the To go further part
left_join(results$dplyr |> select(AIRLINE, exact = arrival_delay),
          on_sample$results$dplyr |> select(AIRLINE, `10% sample` = arrival_delay), by = "AIRLINE")

**Conclusion** The three engines give the same results, on the sample as on the full data. Loading the $5.8$ million flights from the Parquet files takes a few seconds and $1.4$ GB of memory. On the full data, DuckDB is the fastest, then `data.table` and `dplyr` (a few hundred milliseconds each), and `dplyr` allocates the most memory ($470$ MB for its intermediate copies, against $73$ MB for `data.table`, which works in place, and almost nothing on the R side for DuckDB, which computes outside R, with several threads). On the sample, the ranking is reversed, DuckDB being the slowest: with $10$ times more rows, `dplyr` and `data.table` take $5$ to $10$ times longer, as their cost grows with the number of rows, while DuckDB takes at most twice as long, its time on small data being mostly fixed costs (planning the query, converting the results). A comparison of engines therefore depends on the size of the data, and must be made at the right scale (the timings themselves change from one run to another, with the load of the machine). Finally, the means estimated on the sample are within half a minute of the exact ones (e.g. $14.6$ against $14.5$ minutes for Spirit, $-0.65$ against $-0.98$ for Alaska), as predicted in Part 1: for such aggregates, the sample is enough.

---

**Question** *Predict before you run:* is it faster to compute on data already in memory, or directly on the Parquet files? Compute the same statistics with Arrow (`open_dataset`, the same `dplyr` verbs and `collect()`) and with DuckDB (an SQL query on `read_parquet('data/flights_full_parquet/*/*.parquet', hive_partitioning = true)`), directly from the files. Check the results, and compare the times and the memory of the R session with the engines of the previous question.

**Answer**

In [ ]:
query_files <- "SELECT AIRLINE, AVG(ARRIVAL_DELAY) AS arrival_delay, AVG(CAST(ARRIVAL_DELAY > 15 AS DOUBLE)) AS late
                FROM read_parquet('data/flights_full_parquet/*/*.parquet', hive_partitioning = true)
                GROUP BY AIRLINE ORDER BY AIRLINE"
by_airline_arrow_files <- function() open_dataset("data/flights_full_parquet") |> group_by(AIRLINE) |>
  summarise(arrival_delay = mean(ARRIVAL_DELAY, na.rm = TRUE), late = mean(as.numeric(ARRIVAL_DELAY > 15), na.rm = TRUE)) |>
  arrange(AIRLINE) |> collect()
by_airline_duckdb_files <- function() dbGetQuery(con, query_files)
c(`arrow agrees` = isTRUE(all.equal(results$dplyr, as_tibble(by_airline_arrow_files()), check.attributes = FALSE)),
  `duckdb agrees` = isTRUE(all.equal(results$dplyr, as_tibble(by_airline_duckdb_files()), check.attributes = FALSE)))
bench::mark(arrow = by_airline_arrow_files(), duckdb = by_airline_duckdb_files(), iterations = 5, check = FALSE, filter_gc = FALSE) |>
  transmute(engine = as.character(expression), median, mem_alloc)

**Conclusion** Computing directly on the files is the fastest option: DuckDB reads the $12$ Parquet files and computes the statistics faster than any engine on the data already in memory (about $0.1$ s), and Arrow in a few tenths of a second, while the R session only receives the $14$ rows of the result (a few kB). Loading the data first costs a few seconds and $1.4$ GB: when the question is an aggregate, the data do not need to be loaded in R at all. Only the $2$ needed columns are read (columnar format), in parallel, and the binary data need no parsing.

---

**Question** Join the results with the full names of the airlines and plot the mean delay and the proportion of late flights. With DuckDB, display the SQL query generated by `dbplyr` (`show_query()`).

**Answer**

In [ ]:
results$dplyr |> left_join(airlines |> rename(airline_name = AIRLINE), by = c("AIRLINE" = "IATA_CODE")) |>
  mutate(airline_name = fct_reorder(airline_name, late)) |>
  pivot_longer(c(arrival_delay, late), names_to = "statistic") |>
  ggplot(aes(value, airline_name)) + geom_point() + facet_wrap(~ statistic, scales = "free_x") + labs(x = NULL, y = NULL)
flights_db |> group_by(AIRLINE) |> summarise(arrival_delay = mean(ARRIVAL_DELAY, na.rm = TRUE)) |> show_query()
rm(flights_full)  # the next parts use the sample
invisible(gc())

**Conclusion** Spirit (NK) and Frontier (F9), two low-cost airlines, have the largest mean delays ($15$ and $12$ minutes) and the most flights arriving more than $15$ minutes late ($29\%$ and $25\%$), while Alaska, Delta and Hawaiian are on time on average, with $11$ to $13\%$ of late flights. The `dplyr` verbs applied to the DuckDB table are translated into SQL by `dbplyr` (`SELECT AIRLINE, AVG(ARRIVAL_DELAY) ... GROUP BY AIRLINE`) and executed by the database: the same code works on a data frame, a database or (with `sparklyr`) a cluster.

---

## 3. Profiling and optimizing

The cumulated arrival delay along the year can be computed with `cumsum`. To understand where the time goes, we write this function ourselves (PC7, *Complexity of cumulative sums*).

In [ ]:
cumsum_0 <- function(x) {
  res <- numeric(0)
  for (i in seq_along(x)) {
    res[i] <- 0
    for (j in seq_len(i)) res[i] <- res[i] + x[j]
  }
  res
}

**Question** *Predict before you run:* how does the time of `cumsum_0` grow with the length $n$ of `x`? Measure it for $n = 250, 500, 1000, 2000$ and plot it in log-log scale.

**Answer**

In [ ]:
scaling <- tibble(n = c(250, 500, 1000, 2000)) |>
  mutate(time = map_dbl(n, \(n) { x <- rnorm(n); as.numeric(bench::mark(cumsum_0(x), iterations = 3)$median) }))
scaling
ggplot(scaling, aes(n, time)) + geom_point() + geom_line() + scale_x_log10() + scale_y_log10() + labs(y = "time (s)")
coef(lm(log(time) ~ log(n), data = scaling))[2]

**Conclusion** The time of `cumsum_0` grows like $n^2$: the slope in log-log scale is close to $2$ (between $1.8$ and $2$ depending on the run, the small sizes being dominated by constant costs), as expected from the $n(n-1)/2$ additions of the double loop (PC7, *Complexity of cumulative sums*). For the $582\,000$ flights, it would take hours.

---

**Question** Write `cumsum_1` with a single loop (`res[i] <- res[i - 1] + x[i]`) growing the result vector, `cumsum_2` with a preallocated result (`numeric(length(x))`), and compare them with the built-in `cumsum` for $n = 10^5$. Then, if a compiler is available (`pkgbuild::has_build_tools()`), compile the C++ functions of `cumsum.cc` with `Rcpp::sourceCpp` and add them to the comparison.

**Answer**

In [ ]:
cumsum_1 <- function(x) {
  res <- x[1]
  for (i in seq_along(x)[-1]) res[i] <- res[i - 1] + x[i]
  res
}
cumsum_2 <- function(x) {
  res <- numeric(length(x))
  res[1] <- x[1]
  for (i in seq_along(x)[-1]) res[i] <- res[i - 1] + x[i]
  res
}
x <- rnorm(1e5)
candidates <- list(`loop, growing vector` = quote(cumsum_1(x)), `loop, preallocated` = quote(cumsum_2(x)), `built-in cumsum` = quote(cumsum(x)))
if (pkgbuild::has_build_tools()) {
  Rcpp::sourceCpp("cumsum.cc")
  candidates <- c(candidates, list(`Rcpp loop` = quote(cumsum_4(x)), `Rcpp std::partial_sum` = quote(cumsum_5(x))))
}
timings <- bench::mark(exprs = candidates, iterations = 10, check = TRUE, filter_gc = FALSE)
timings |> transmute(version = names(candidates), median, mem_alloc, `relative to cumsum` = as.numeric(median) / as.numeric(median[3]))

**Conclusion** With a single loop, the algorithm is linear, but its R implementation is still $10$ to $15$ times slower than the built-in `cumsum` when the result is preallocated, and about $40$ times slower when the vector grows at each step (which also allocates $8$ times more memory; recent versions of R grow vectors in place by blocks, which limits the damage compared with the theoretical $O(n^2)$ copies). The compiled C++ loop (`Rcpp`) is as fast as the built-in `cumsum` (itself written in C), and `std::partial_sum` is even a little faster. The order of the optimizations matters: first the algorithm ($O(n^2) \to O(n)$), then the memory management, then the language.

---

**Question** Profile the computation of the cumulated delay along the year (sort the flights by date and departure time, compute the cumulated sum, build the dates, plot) with `profvis`. Which step takes the most time? Is `cumsum` worth optimizing?

**Answer**

In [ ]:
profvis::profvis({
  timeline <- flights |> arrange(MONTH, DAY, SCHEDULED_DEPARTURE) |>
    mutate(cumulated_delay = cumsum(replace_na(ARRIVAL_DELAY, 0)),
           date = make_datetime(YEAR, MONTH, DAY, as.integer(substr(SCHEDULED_DEPARTURE, 1, 2)), as.integer(substr(SCHEDULED_DEPARTURE, 3, 4))))
  print(ggplot(timeline, aes(date, cumulated_delay / 60)) + geom_line() + labs(y = "cumulated delay (hours)"))
})

**Conclusion** The profile shows that `cumsum` takes almost no time (a few milliseconds): most of the time goes to drawing the plot (more than $1$ s for half a million points), then to sorting the flights and building the dates (a few tenths of a second each). Optimizing `cumsum` would be useless here: a profiler shows where the time actually goes, which is rarely where we expect. The cumulated delay reaches about $41\,000$ hours over the year for the $10\%$ sample.

---

## 4. Parallel cross validation

We want to predict the arrival delay from the information available at the departure. We compare $5$ linear models by an $8$-fold cross validation (note: in a formula, `DEPARTURE_DELAY/SCHEDULED_TIME` means *nesting*, not a ratio: the ratio is written `I(DEPARTURE_DELAY / SCHEDULED_TIME)`).

In [ ]:
flights_model_data <- flights |> select(ARRIVAL_DELAY, DEPARTURE_DELAY, SCHEDULED_TIME, AIRLINE) |> drop_na()
formulas <- c("ARRIVAL_DELAY ~ DEPARTURE_DELAY",
              "ARRIVAL_DELAY ~ DEPARTURE_DELAY + SCHEDULED_TIME",
              "ARRIVAL_DELAY ~ DEPARTURE_DELAY + SCHEDULED_TIME + I(DEPARTURE_DELAY / SCHEDULED_TIME)",
              "ARRIVAL_DELAY ~ DEPARTURE_DELAY + SCHEDULED_TIME + I(DEPARTURE_DELAY / SCHEDULED_TIME) + AIRLINE",
              "ARRIVAL_DELAY ~ (DEPARTURE_DELAY + SCHEDULED_TIME + I(DEPARTURE_DELAY / SCHEDULED_TIME)) * AIRLINE")
set.seed(2)
folds <- sample(rep(1:8, length.out = nrow(flights_model_data)))
cv_rmse <- function(formula, fold, data, folds) {
  fit <- lm(as.formula(formula), data = data[folds != fold, ])
  sqrt(mean((predict(fit, data[folds == fold, ]) - data$ARRIVAL_DELAY[folds == fold])^2))
}
tasks <- expand_grid(formula = formulas, fold = 1:8)

**Question** Compute the $40$ cross validation errors sequentially with `map2_dbl`, and display the mean CV RMSE of each model. Which model would you choose?

**Answer**

In [ ]:
sequential_time <- system.time(
  tasks$rmse <- map2_dbl(tasks$formula, tasks$fold, cv_rmse, data = flights_model_data, folds = folds)
)[["elapsed"]]
tasks |> group_by(formula) |> summarise(`CV RMSE` = mean(rmse), `fold SD` = sd(rmse)) |>
  mutate(model = match(formula, formulas), .before = 1) |> arrange(model) |> select(-formula)
paired <- tasks |> mutate(model = match(formula, formulas)) |> filter(model %in% 4:5) |>
  select(fold, model, rmse) |> pivot_wider(names_from = model, values_from = rmse, names_prefix = "m")
summary(paired$m5 - paired$m4)
sequential_time

**Conclusion** The CV RMSE decreases from $12.9$ minutes (departure delay only) to $12.7$ with the scheduled time and the ratio, and to $12.5$ with the airline effects (models $4$ and $5$, similar to the displayed precision): the arrival delay is mostly the departure delay, and the other variables only bring small corrections. The paired differences compare both models on each validation fold. Their spread is descriptive because the training sets overlap. Model $5$ reduced fold RMSE by $0.017$ to $0.031$ minutes (mean $0.024$, about $1.4$ seconds). We choose model $4$ for its simpler specification.

---

**Question** Start $4$ background R processes with `mirai::daemons(4)`, and run the same computation in parallel with `map2_dbl(..., .f = in_parallel(\(formula, fold) ..., cv_rmse = cv_rmse, data = flights_model_data, folds = folds))`: the function sent to the workers must be self-contained, its objects being passed explicitly. Compare the times. Then measure the time of a parallel computation whose tasks are tiny (e.g. the mean of $100$ numbers, $40$ times). Explain the results with PC7, *Speed-up laws*.

**Answer**

In [ ]:
mirai::daemons(4)
parallel_time <- system.time(
  parallel_rmse <- map2_dbl(tasks$formula, tasks$fold,
                            in_parallel(\(formula, fold) cv_rmse(formula, fold, data, folds),
                                        cv_rmse = cv_rmse, data = flights_model_data, folds = folds))
)[["elapsed"]]
tiny <- bench::mark(sequential = map_dbl(1:40, \(i) mean(rnorm(100))),
                    parallel = map_dbl(1:40, in_parallel(\(i) mean(rnorm(100)))), iterations = 5, check = FALSE)
mirai::daemons(0)
c(`same results` = isTRUE(all.equal(parallel_rmse, tasks$rmse)), sequential = sequential_time, `parallel (4 workers)` = parallel_time,
  `speed-up` = sequential_time / parallel_time)
tiny |> select(expression, median)

**Conclusion** With $4$ workers, the cross validation is $2$ to $2.6$ times faster (about $14$ s instead of $30$ to $35$ s), rather than $4$ times: sending the data to the workers, starting the tasks and collecting the results cost time, and the tasks are not all of the same length. For the $40$ tiny tasks, the parallel version is $6$ to $8$ times *slower* than the sequential one (about $7$ ms against $1$ ms): the overhead of each task is much larger than its computation. Parallelism pays off only for tasks that are long compared with its overhead (PC7, *Speed-up laws*, with an overhead per worker).

---

## 5. Serving a model

The chosen model has to be made available: to people through a web application, to other programs through a REST API.

**Question** Fit the fourth model on all the data. What is its size in memory? Which of its components take the space? Reduce it with `butcher::butcher()`; then remove by hand the components that `predict` does not need (the $QR$ decomposition matrix `qr$qr`, the `effects`, the `residuals` and the `model` data frame). Check that its predictions are unchanged, and save it in `flights_model.rds`.

**Answer**

In [ ]:
flights_model <- lm(ARRIVAL_DELAY ~ DEPARTURE_DELAY + SCHEDULED_TIME + I(DEPARTURE_DELAY / SCHEDULED_TIME) + AIRLINE, data = flights_model_data)
sizes <- map_dbl(flights_model, \(component) as.numeric(lobstr::obj_size(component)))
head(sort(sizes, decreasing = TRUE), 5) / 1e6
butchered <- butcher::butcher(flights_model)
small_model <- butchered
small_model$qr$qr <- NULL                                  # predict only needs the rank and the pivot of the QR decomposition
small_model[c("effects", "residuals", "model")] <- NULL
newdata <- tibble(DEPARTURE_DELAY = c(0, 30, 120), SCHEDULED_TIME = c(60, 180, 300), AIRLINE = c("AA", "DL", "WN"))
saveRDS(small_model, "flights_model.rds")
c(`full model (MB)` = as.numeric(lobstr::obj_size(flights_model)) / 1e6, `butchered model (MB)` = as.numeric(lobstr::obj_size(butchered)) / 1e6,
  `reduced by hand (MB)` = as.numeric(lobstr::obj_size(small_model)) / 1e6,
  `same predictions` = isTRUE(all.equal(predict(flights_model, newdata), predict(small_model, newdata))))

**Conclusion** The fitted linear model takes $119$ MB, for $16$ coefficients: it stores the $QR$ decomposition of the design matrix ($78$ MB), the model frame ($23$ MB, a copy of the data), the effects, the residuals and the fitted values ($5$ to $9$ MB each). `butcher` only removes the components it knows to be useless for any model of this class (here little: $114$ MB). Removing by hand everything that `predict` does not need gives $11$ kB, with exactly the same predictions: the model file to deploy is $10\,000$ times smaller (and does not contain the training data, which also matters for privacy).

---

The file `flights_api.R` describes a REST API with `plumber2`: comments `#*` above a function declare the route (`@get /predict`), the expected query parameters and their types, and the format of the answer.

In [ ]:
cat(readLines("flights_api.R"), sep = "\n")

**Question** Start the API in a background R process (`callr::r_bg`), wait until it answers, and query it with `httr2` for a flight of American Airlines with a departure delay of $30$ minutes and a scheduled time of $180$ minutes. Compare with `predict`. Then write a `testthat` test of the API that does not need a running server (`plumber2::api("flights_api.R")$test_request(fiery::fake_request(url))`).

**Answer**

In [ ]:
server <- callr::r_bg(\() plumber2::api("flights_api.R") |> plumber2::api_run(port = 8765, block = TRUE, showcase = FALSE))
request <- httr2::request("http://127.0.0.1:8765/predict") |>
  httr2::req_url_query(departure_delay = 30, scheduled_time = 180, airline = "AA") |>
  httr2::req_retry(max_tries = 20, retry_on_failure = TRUE, backoff = \(i) 1)
answer <- httr2::resp_body_json(httr2::req_perform(request))
server$kill()
c(API = answer$arrival_delay, predict = unname(predict(small_model, tibble(DEPARTURE_DELAY = 30, SCHEDULED_TIME = 180, AIRLINE = "AA"))))

In [ ]:
library(testthat)
test_that("the API predicts the arrival delay and rejects unknown airlines", {
  api <- plumber2::api("flights_api.R")
  query <- function(url) {
    body <- api$test_request(fiery::fake_request(url))$body
    jsonlite::fromJSON(if (is.raw(body)) rawToChar(body) else body)
  }
  ok <- query("http://localhost/predict?departure_delay=30&scheduled_time=180&airline=AA")
  expect_equal(ok$arrival_delay, unname(predict(small_model, tibble(DEPARTURE_DELAY = 30, SCHEDULED_TIME = 180, AIRLINE = "AA"))), tolerance = 1e-4)  # JSON keeps 4 significant digits
  expect_equal(query("http://localhost/predict?departure_delay=30&scheduled_time=180&airline=XX")$error, "unknown airline")
})

**Conclusion** The API started in a background process answers $24.46$ minutes for an American Airlines flight leaving $30$ minutes late for a $3$-hour flight, the prediction of the model (the JSON answer keeps $4$ significant digits). The test does not need a running server: `test_request` sends a fake request through the API object, which makes it fast and usable in continuous integration. It checks both a correct answer and the error message for an unknown airline: an API must be tested on its failures too.

---

The file `flights_app.R` is a `shiny` application, with a `bslib` layout, using the same model: run it from RStudio (*Run App*) or with `shiny::runApp("flights_app.R")`.

In [ ]:
cat(readLines("flights_app.R"), sep = "\n")

## About this lab

**Use of AI.** Artificial intelligence tools have been used to revise the content of this lab. The main contributor takes full responsibility for this content.

**License.** The texts of this lab are under the [Creative Commons Attribution-ShareAlike 4.0](https://creativecommons.org/licenses/by-sa/4.0/) license (CC BY-SA 4.0), and its code under the [MIT](https://opensource.org/license/mit) license; the datasets keep their own licenses. Main contributor: E. Le Pennec. Contributors: S. Boucheron, A. Dieuleveut, A.K. Fermin, S. Gadat, S. Gaiffas, B. Goujaud, A. Guilloux, Ch. Keribin, Th. Klein, E. Matzner, J. Read, M. Sangnier, E. Scornet.